In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import sentencepiece as spm

import os
import re
import math
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [3]:
# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_path = os.path.join(os.getcwd(), '..', '..', 'work', 'data', 'ex08')
data_path = os.path.join(base_path, 'ChatbotData.csv')

In [ ]:
# --- [특수 토큰 및 환경 상수 정의] ---
PAD_TOKEN = "[PAD]"  # 0
UNK_TOKEN = "[UNK]"  # 1
BOS_TOKEN = "[BOS]"  # 2 (Start of Sentence)
EOS_TOKEN = "[EOS]"  # 3 (End of Sentence)

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]
MAX_LEN = 40  # 분석 결과 기반 적절한 시퀀스 최대 길이

class SentencePieceTokenizer:
    def __init__(self, model_prefix='spm_chatbot', vocab_size=8000):
        self.model_prefix = model_prefix
        self.vocab_size = vocab_size
        self.model_path = f"{model_prefix}.model"
        self.sp = spm.SentencePieceProcessor()
        
    def train_tokenizer(self, corpus_path):
        """텍스트 코퍼스 파일을 기반으로 SentencePiece 모델을 학습합니다."""
        print("Creating and training SentencePiece tokenizer...")
        spm.SentencePieceTrainer.train(
            input=corpus_path,
            model_prefix=self.model_prefix,
            vocab_size=self.vocab_size,
            character_coverage=1.0,
            model_type='bpe',  # Byte Pair Encoding 방식 적용
            max_sentence_length=999999,
            pad_id=0, unk_id=1, bos_id=2, eos_id=3,
            pad_piece=PAD_TOKEN, unk_piece=UNK_TOKEN,
            bos_piece=BOS_TOKEN, eos_piece=EOS_TOKEN
        )
        self.load_tokenizer()
        
    def load_tokenizer(self):
        """학습된 모델 파일을 프로세서에 로드합니다."""
        if os.path.exists(self.model_path):
            self.sp.load(self.model_path)
            print(f"Successfully loaded SentencePiece model from {self.model_path}")
        else:
            raise FileNotFoundError(f"Model file not found at {self.model_path}. Train the tokenizer first.")
            
    def encode(self, text, add_bos=False, add_eos=False):
        """문장을 인덱스 리스트로 변환합니다."""
        tokens = self.sp.encode_as_ids(text)
        if add_bos:
            tokens = [2] + tokens
        if add_eos:
            tokens = tokens + [3]
        return tokens
        
    def decode(self, ids):
        """인덱스 리스트를 다시 문장으로 복원합니다."""
        # 패딩이나 특수 토큰 인덱스를 필터링하여 복원율 향상
        clean_ids = [int(i) for i in ids if i not in [0, 2, 3]]
        return self.sp.decode_ids(clean_ids)

    def get_vocab_size(self):
        return self.vocab_size


# --- [기능 2: 데이터 전처리 및 데이터셋 구축] ---
class ChatbotDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_len=40):
        self.df = pd.read_csv(data_path)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # 간단한 정규식 전처리
        self.df['Q'] = self.df['Q'].apply(self.clean_text)
        self.df['A'] = self.df['A'].apply(self.clean_text)
        
        self.inputs, self.dec_inputs, self.targets = self.preprocess()
        
    def clean_text(self, text):
        text = re.sub(r"([?.!,])", r" \1 ", text)
        text = text.strip()
        return text
        
    def preprocess(self):
        inputs, dec_inputs, targets = [], [], []
        
        for q, a in zip(self.df['Q'], self.df['A']):
            # 인코더 입력: 질문
            enc_input = self.tokenizer.encode(q, add_bos=False, add_eos=False)
            # 디코더 입력: [BOS] + 답변
            dec_input = self.tokenizer.encode(a, add_bos=True, add_eos=False)
            # 디코더 타겟: 답변 + [EOS]
            target = self.tokenizer.encode(a, add_bos=False, add_eos=True)
            
            # 패딩 및 최대 길이 맞춤 (Truncate & Post-Padding)
            enc_input = enc_input[:self.max_len] + [0] * max(0, self.max_len - len(enc_input))
            dec_input = dec_input[:self.max_len] + [0] * max(0, self.max_len - len(dec_input))
            target = target[:self.max_len] + [0] * max(0, self.max_len - len(target))
            
            inputs.append(enc_input)
            dec_inputs.append(dec_input)
            targets.append(target)
            
        return (torch.tensor(inputs, dtype=torch.long), 
                torch.tensor(dec_inputs, dtype=torch.long), 
                torch.tensor(targets, dtype=torch.long))
                
    def __len__(self):
        return len(self.inputs)
        
    def __getitem__(self, idx):
        return self.inputs[idx], self.dec_inputs[idx], self.targets[idx]


# --- [기능 3: 손실 함수 정의 (Padding Mask 반영 필수)] ---
def loss_function(real, pred):
    """패딩 토큰(0)을 제외하고 손실을 계산하는 마스크드 크로스엔트로피 손실 함수"""
    # pred shape: (batch_size, seq_len, vocab_size) -> 변환 필요
    # real shape: (batch_size, seq_len)
    cost = F.cross_entropy(pred.view(-1, pred.size(-1)), real.view(-1), reduction='none')
    
    # 패딩 위치는 0, 실제 데이터 위치는 1인 마스크 생성
    mask = (real.view(-1) != 0).float()
    cost = cost * mask
    
    return cost.sum() / mask.sum()


# --- [기능 4: 모델 학습 함수] ---
def train_transformer(model, dataloader, epochs=10, lr=0.0001):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    train_losses = []

    print("\n========= 🚀 학습을 개시합니다 =========")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_idx, (inputs, dec_inputs, targets) in enumerate(dataloader):
            inputs, dec_inputs, targets = inputs.to(device), dec_inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            
            # Forward 연산 (Teacher Forcing 구조)
            # Transformer 내부에서 마스크들이 동적으로 생성됨
            outputs = model(inputs, dec_inputs)
            
            loss = loss_function(targets, outputs)
            loss.backward()
            
            # 그래디언트 클리핑으로 발산 방지
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            train_losses.append(total_loss)
            
            
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1:02d}/{epochs} | Avg Masked Loss: {avg_loss:.4f}")
    
    return train_losses


def train_transformer_with_train_earlystop(model, dataloader, epochs=20, lr=0.0001, patience=3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    
    # Train Loss 기준으로 조기 종료를 판단할 인스턴스 생성
    early_stopping = EarlyStopping(patience=patience, verbose=True)
    
    print("\n========= 🚀 Train 수렴 기준 Early Stopping 학습 시작 =========")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_idx, (inputs, dec_inputs, targets) in enumerate(dataloader):
            inputs, dec_inputs, targets = inputs.to(device), dec_inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs, dec_inputs)
            loss = loss_function(targets, outputs)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_train_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1:02d}/{epochs} | Avg Masked Train Loss: {avg_train_loss:.4f}")
        
        # 💡 검증셋이 없으므로, 현재 에포크의 Train Loss를 그대로 매개변수로 전달합니다.
        early_stopping(avg_train_loss, model)
        
        if early_stopping.early_stop:
            print(f"🛑 조기 종료: Train Loss가 {patience} 에포크 동안 더 이상 갱신되지 않아 최적 수렴으로 판단, {epoch+1} 에포크에서 종료합니다.")
            break
            
    # 학습이 끝난 후 가장 손실이 낮았던 최적의 상태 복원
    if early_stopping.best_model_state is not None:
        model.load_state_dict(early_stopping.best_model_state)
        print(f"🎯 역대 최저 Train Loss({early_stopping.val_loss_min:.4f}) 시점의 가중치로 최종 복원 완료.")

# --- [기능 5: 평가 및 대답 예측 함수 (Autoregressive Inference)] ---
def evaluate_predict(sentence, model, tokenizer, max_len=40):
    """입력된 임의의 문장에 대해 트랜스포머가 한 토큰씩 예측하여 답변을 완성합니다."""
    model.eval()
    
    # 1. 입력 문장 전처리 및 인코더 토큰화
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence).strip()
    enc_input = tokenizer.encode(sentence, add_bos=False, add_eos=False)
    enc_input = enc_input[:max_len] + [0] * max(0, max_len - len(enc_input))
    enc_input = torch.tensor([enc_input], dtype=torch.long).to(device) # Batch 차원 추가
    
    # 2. 디코더 입력 초기화: [BOS] 토큰부터 생성 시작
    dec_input = torch.tensor([[2]], dtype=torch.long).to(device) # shape: (1, 1)
    
    # 3. 예측 루프 시작
    for i in range(max_len):
        with torch.no_grad():
            # 트랜스포머 추론 수행 (현재까지 쌓인 dec_input 활용)
            outputs = model(enc_input, dec_input)
            
            # 가장 마지막 시점(타임스텝)의 로짓을 선택하여 다음 토큰 예측
            predictions = outputs[:, -1, :]
            predicted_id = torch.argmax(predictions, dim=-1).item()
            
            # 만약 끝을 알리는 [EOS](3) 토큰이 나오면 루프 탈출
            if predicted_id == 3:
                break
                
            # 예측된 토큰을 디코더 다음 입력 뒤에 결합(Concat)
            next_token_tensor = torch.tensor([[predicted_id]], dtype=torch.long).to(device)
            dec_input = torch.cat([dec_input, next_token_tensor], dim=1)
            
    # 4. 최종 예측된 인덱스 텐서를 문장으로 디코딩
    generated_ids = dec_input.squeeze(0).tolist()
    answer = tokenizer.decode(generated_ids)
    return answer

def plot_loss_curve(train_losses, val_losses=None, title="Transformer Chatbot Loss Curve"):
    """
    학습 및 검증 손실(Loss) 추이를 선그래프로 시각화합니다.
    
    Args:
        train_losses (list): 에포크별 Train Loss 리스트
        val_losses (list, optional): 에포크별 Validation Loss 리스트. 생략 가능.
        title (str): 그래프의 메인 제목
    """
    # 1. 스타일 및 피규어 크기 설정
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_losses) + 1)
    
    # 2. Train Loss 플로팅
    plt.plot(epochs, train_losses, fontlabel='Train Loss', color='royalblue', 
             linestyle='-', marker='o', linewidth=2, label='Train Loss')
    
    # 3. Validation Loss 플로팅 (데이터가 존재할 때만 그림)
    if val_losses is not None and len(val_losses) > 0:
        plt.plot(epochs, val_losses, fontlabel='Val Loss', color='tomato', 
                 linestyle='--', marker='s', linewidth=2, label='Val Loss')
    
    # 4. 그래프 디테일 설정 (라벨, 타이틀, 격자 등)
    plt.title(title, fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Epochs', fontsize=12, labelpad=10)
    plt.ylabel('Masked Cross Entropy Loss', fontsize=12, labelpad=10)
    
    # x축 눈금을 정수(Epoch) 단위로만 표시
    plt.xticks(epochs)
    
    # 범례 및 그리드 추가
    plt.legend(fontsize=11, loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    # 레이아웃 정렬 후 출력
    plt.tight_layout()
    plt.show()

class EarlyStopping:
    """검증 손실(Val Loss)이 개선되지 않으면 학습을 조기 종료하는 클래스"""
    def __init__(self, patience=3, verbose=True, delta=0):
        self.patience = patience    # 개선이 안 될 때 몇 에포크까지 버틸 것인가
        self.verbose = verbose      # 로그 출력 여부
        self.counter = 0            # 정체 에포크 카운터
        self.best_loss = None       # 역대 최저 검증 손실
        self.early_stop = False     # 조기 종료 플래그
        self.val_loss_min = np.Inf
        self.delta = delta          # 개선되었다고 판단할 최소 변화량
        self.best_model_state = None # 최적의 가중치를 저장할 공간

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_loss is None:
            self.best_loss = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_loss + self.delta:
            self.counter += 1
            if self.verbose:
                print(f" EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """검증 손실이 감소하면 모델의 최적 가중치 상태를 메모리에 백업"""
        if self.verbose:
            print(f" Train loss 감소함 ({self.val_loss_min:.6f} --> {val_loss:.6f}). 가중치 백업 중...")
        
        import copy
        self.best_model_state = copy.deepcopy(model.state_dict())
        self.val_loss_min = val_loss



In [26]:

def create_padding_mask(x):
    # x == 0 위치를 찾아 float형 1로 변환
    mask = (x == 0).float()
    # (batch_size, seq_len) -> (batch_size, 1, 1, seq_len)
    mask = mask.unsqueeze(1).unsqueeze(2)
    return mask

def create_look_ahead_mask(x):
    seq_len = x.size(1)

    # (seq_len, seq_len) 크기의 하삼각 행렬(tril) 생성 후 1에서 빼서
    # 상삼각이 1, 하삼각(자기 자신 포함)이 0이 되도록 설정
    # => 미래 토큰(자신 인덱스보다 큰 위치) 마스킹
    look_ahead_mask = 1 - torch.tril(torch.ones((seq_len, seq_len)))

    # 패딩 마스크 생성 (shape: (batch_size, 1, 1, seq_len))
    padding_mask = create_padding_mask(x)

    # look_ahead_mask: (seq_len, seq_len) -> (1, seq_len, seq_len)
    look_ahead_mask = look_ahead_mask.unsqueeze(0)
    # -> (1, seq_len, seq_len) -> (1, 1, seq_len, seq_len)
    look_ahead_mask = look_ahead_mask.unsqueeze(1)
    look_ahead_mask = look_ahead_mask.to(x.device)

    # look-ahead 마스크와 패딩 마스크를 합성 (둘 중 하나라도 1이면 마스킹)
    # 최종 shape은 브로드캐스팅으로 (batch_size, 1, seq_len, seq_len)
    combined_mask = torch.max(look_ahead_mask, padding_mask)
    return combined_mask

def scaled_dot_product_attention(query, key, value, mask=None):

    # 1) Q와 K의 내적을 통해 score(유사도) 계산
    # key.transpose(-1, -2): (batch_size, heads, depth, seq_len)
    # matmul 결과 shape: (batch_size, heads, seq_len, seq_len)
    matmul_qk = torch.matmul(query, key.transpose(-1, -2))

    # 2) depth에 따라 정규화
    depth = key.size(-1)  # depth = d_model / heads
    logits = matmul_qk / math.sqrt(depth)

    # 3) 마스크가 주어졌다면 -1e9(아주 작은 값)를 더해 소프트맥스에서 제외시키도록 함
    if mask is not None:
        # 텐서플로우: logits += (mask * -1e9)
        # 파이토치 동일 적용
        logits = logits + (mask * -1e9)

    # 4) 소프트맥스 계산해 attention weights 생성
    attention_weights = F.softmax(logits, dim=-1)

    # 5) attention weights와 value의 내적
    output = torch.matmul(attention_weights, value)

    return output, attention_weights


class PositionalEncoding(nn.Module):
    def __init__(self, position, d_model):
        super(PositionalEncoding, self).__init__()
        self.d_model = d_model
        self.position = position

        self.pos_encoding = self._build_pos_encoding(position, d_model)

    def _get_angles(self, position, i, d_model):
        return 1.0 / (10000.0 ** ((2.0 * (i // 2)) / d_model)) * position

    def _build_pos_encoding(self, position, d_model):
        pos = torch.arange(position, dtype=torch.float32).unsqueeze(1)
        i = torch.arange(d_model, dtype=torch.float32).unsqueeze(0)

        angle_rads = self._get_angles(pos, i, d_model)
        sines = torch.sin(angle_rads[:, 0::2])
        cosines = torch.cos(angle_rads[:, 1::2])

        pos_encoding = torch.zeros(position, d_model)
        pos_encoding[:, 0::2] = sines
        pos_encoding[:, 1::2] = cosines

        pos_encoding = pos_encoding.unsqueeze(0)  # shape: [1, position, d_model]
        return pos_encoding

    def forward(self, x):
        return x + self.pos_encoding[:, :x.size(1), :].to(x.device)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, name="multi_head_attention"):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model은 num_heads로 나누어떨어져야 함
        assert d_model % num_heads == 0

        self.depth = d_model // num_heads

        # 파이토치에서 Dense는 nn.Linear로 대응
        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)

        self.out_dense = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        """
        x: (batch_size, seq_len, d_model)
        => (batch_size, num_heads, seq_len, depth) 형태로 변환
        """
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)  # (batch_size, num_heads, seq_len, depth)
        return x

    def forward(self, query, key, value, mask=None):
        """
        query, key, value: (batch_size, seq_len, d_model)
        mask: (batch_size, 1, seq_len, seq_len) 등으로 broadcast 가능하도록 구성
        """
        batch_size = query.size(0)

        # Q, K, V에 각각 Linear 적용
        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        # Head 분할
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        # 스케일드 닷 프로덕트 어텐션
        scaled_attention, _ = scaled_dot_product_attention(query, key, value, mask)

        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        scaled_attention = scaled_attention.permute(0, 2, 1, 3).contiguous()

        # 다시 (batch_size, seq_len, d_model)로 합치기
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)

        # 최종 Dense
        output = self.out_dense(concat_attention)
        return output


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)  # 이전에 구현한 MHA
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)

        # 피드포워드 부분 (Dense -> ReLU -> Dense)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, mask=None):
        # (1) 멀티 헤드 어텐션 (셀프 어텐션)
        attn_output = self.mha(x, x, x, mask)  # (batch_size, seq_len, d_model)
        attn_output = self.dropout1(attn_output)
        out1 = self.norm1(x + attn_output)     # 잔차 연결 + LayerNorm

        # (2) 피드포워드 신경망
        ffn_output = self.ffn(out1)            # (batch_size, seq_len, d_model)
        ffn_output = self.dropout2(ffn_output)
        out2 = self.norm2(out1 + ffn_output)   # 잔차 연결 + LayerNorm

        return out2


class Encoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,
                 ff_dim,
                 d_model,
                 num_heads,
                 dropout=0.1):
        super(Encoder, self).__init__()
        self.d_model = d_model

        # (1) 임베딩 레이어
        self.embedding = nn.Embedding(vocab_size, d_model)

        # (2) 포지셔널 인코딩
        self.pos_encoding = PositionalEncoding(position=vocab_size, d_model=d_model)

        self.dropout = nn.Dropout(dropout)

        # (3) EncoderLayer 쌓기
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        # (1) 임베딩 & sqrt(d_model)로 스케일링
        x = self.embedding(x) * math.sqrt(self.d_model)

        # (2) 포지셔널 인코딩 적용 + 드롭아웃
        x = self.pos_encoding(x)  # shape: (batch_size, seq_len, d_model)
        x = self.dropout(x)

        # (3) num_layers만큼 쌓아올린 EncoderLayer 통과
        for layer in self.enc_layers:
            x = layer(x, mask)

        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(DecoderLayer, self).__init__()

        # 첫 번째 서브 레이어 (디코더 내부 셀프 어텐션)
        self.self_mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)

        # 두 번째 서브 레이어 (인코더-디코더 어텐션)
        self.encdec_mha = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)

        # 세 번째 서브 레이어 (피드포워드 네트워크)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),  # Dense(units=ff_dim)
            nn.ReLU(),                   # activation='relu'
            nn.Linear(ff_dim, d_model)   # Dense(units=d_model)
        )
        self.norm3 = nn.LayerNorm(d_model, eps=1e-6)

        # 드롭아웃
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_outputs, look_ahead_mask=None, padding_mask=None):
        # 1) 셀프 어텐션 (디코더 내부)
        self_attn_out = self.self_mha(x, x, x, mask=look_ahead_mask)
        self_attn_out = self.dropout1(self_attn_out)
        out1 = self.norm1(x + self_attn_out)  # 잔차 연결 + LayerNorm

        # 2) 인코더-디코더 어텐션
        encdec_attn_out = self.encdec_mha(out1, enc_outputs, enc_outputs, mask=padding_mask)
        encdec_attn_out = self.dropout2(encdec_attn_out)
        out2 = self.norm2(out1 + encdec_attn_out)  # 잔차 연결 + LayerNorm

        # 3) 피드포워드 (Dense -> ReLU -> Dense)
        ffn_out = self.ffn(out2)
        ffn_out = self.dropout3(ffn_out)
        out3 = self.norm3(out2 + ffn_out)  # 잔차 연결 + LayerNorm

        return out3
    
class Decoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,
                 ff_dim,
                 d_model,
                 num_heads,
                 dropout=0.1):
        super(Decoder, self).__init__()
        self.d_model = d_model

        # (1) 임베딩 레이어
        self.embedding = nn.Embedding(vocab_size, d_model)

        # (2) 포지셔널 인코딩
        # 실제 학습 시에는 최대 시퀀스 길이에 맞추어 쓰기도 함
        self.pos_encoding = PositionalEncoding(position=vocab_size, d_model=d_model)

        self.dropout = nn.Dropout(dropout)

        # (3) DecoderLayer 쌓기
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, enc_outputs, look_ahead_mask=None, padding_mask=None):
        # (1) 임베딩 + sqrt(d_model)로 스케일링
        x = self.embedding(x) * math.sqrt(self.d_model)

        # (2) 포지셔널 인코딩 + 드롭아웃
        x = self.pos_encoding(x)    # (batch_size, tgt_seq_len, d_model)
        x = self.dropout(x)

        # (3) num_layers만큼 쌓인 DecoderLayer 통과
        for layer in self.dec_layers:
            x = layer(x, enc_outputs, look_ahead_mask, padding_mask)

        return x
    

class Transformer(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,      # 인코더/디코더 층 수
                 units,           # feed-forward 네트워크의 중간 차원(ff_dim)
                 d_model,         # 임베딩 및 내부 표현 차원
                 num_heads,       # 멀티헤드 어텐션의 헤드 수
                 dropout=0.1):
        super(Transformer, self).__init__()

        # 인코더
        self.encoder = Encoder(
            vocab_size=vocab_size,
            num_layers=num_layers,
            ff_dim=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        # 디코더
        self.decoder = Decoder(
            vocab_size=vocab_size,
            num_layers=num_layers,
            ff_dim=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        # 최종 출력층: (d_model) -> (vocab_size)
        self.final_linear = nn.Linear(d_model, vocab_size)

        # 참고: 텐서플로우 코드의 `name="transformer"`는 파이토치에선 보통 사용 안 함

    def forward(self, inputs, dec_inputs):
        # 1) 인코더 패딩 마스크 생성
        enc_padding_mask = create_padding_mask(inputs)     # shape (batch_size, 1, 1, src_seq_len)

        # 2) 디코더 look-ahead + 패딩 마스크
        look_ahead_mask = create_look_ahead_mask(dec_inputs)  # shape (batch_size, 1, tgt_seq_len, tgt_seq_len)

        # 3) 디코더에서 인코더 출력 쪽을 마스킹할 때 쓸 패딩 마스크
        dec_padding_mask = create_padding_mask(inputs)        # shape (batch_size, 1, 1, src_seq_len)

        # 4) 인코더 수행
        enc_outputs = self.encoder(
            x=inputs,
            mask=enc_padding_mask
        )  # shape: (batch_size, src_seq_len, d_model)

        # 5) 디코더 수행
        dec_outputs = self.decoder(
            x=dec_inputs,           # (batch_size, tgt_seq_len)
            enc_outputs=enc_outputs,# (batch_size, src_seq_len, d_model)
            look_ahead_mask=look_ahead_mask,
            padding_mask=dec_padding_mask
        )  # shape: (batch_size, tgt_seq_len, d_model)

        # 6) 최종 Dense (vocab_size)
        logits = self.final_linear(dec_outputs)  # (batch_size, tgt_seq_len, vocab_size)
        return logits



In [ ]:
# execution

# 예: 하이퍼파라미터 설정
NUM_LAYERS = 2     # 인코더/디코더 층 수
D_MODEL = 256      # 임베딩 및 내부 표현 차원
NUM_HEADS = 8      # 멀티헤드 어텐션에서의 헤드 수
UNITS = 512        # 피드포워드 신경망의 은닉 차원
DROPOUT = 0.1      # 드롭아웃 비율
VOCAB_SIZE = 8000 # 단어 집합 크기(예시)

# --- [Step 1: SentencePiece 코퍼스 추출 파일 준비] ---
# SentencePiece는 별도의 순수 텍스트(txt) 학습용 파일이 필요합니다.
df_raw = pd.read_csv(data_path)
corpus_txt_path = os.path.join(base_path, 'corpus.txt')

with open(corpus_txt_path, 'w', encoding='utf-8') as f:
    for text in df_raw['Q'].tolist() + df_raw['A'].tolist():
        f.write(text + '\n')
        
# --- [Step 2: 토크나이저 빌드] ---
vocab_size = 4000  # 데이터 크기가 아담하므로 적정한 크기로 압축
tokenizer = SentencePieceTokenizer(model_prefix='spm_bpe_chatbot', vocab_size=vocab_size)
tokenizer.train_tokenizer(corpus_txt_path)

# --- [Step 3: 데이터셋 및 데이터로더 생성] ---
dataset = ChatbotDataset(data_path, tokenizer, max_len=MAX_LEN)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# --- [Step 4: 트랜스포머 인스턴스화 및 학습] ---
# 파라미터는 편의에 맞춰 튜닝하여 사용하세요.
transformer_bot = Transformer(
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
    units=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT
)
train_transformer(transformer_bot, dataloader, epochs=10, lr=0.0005)

# --- [Step 5: 예측 함수를 통한 모델 평가 및 테스트] ---
print("\n========= 🤖 챗봇 서비스 추론 테스트 =========")
test_questions = [
    "오늘 너무 힘들다",
    "10년 연애 끝 부질없네",
    "인사해줘",
    "맛있는 거 추천해줘"
]

for question in test_questions:
    response = evaluate_predict(question, transformer_bot, tokenizer, max_len=MAX_LEN)
    print(f"질문(Q): {question}")
    print(f"답변(A): {response}")
    print("-" * 40)

Creating and training SentencePiece tokenizer...
Successfully loaded SentencePiece model from spm_bpe_chatbot.model


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /home/ysoh1113/workspace/projects/AIFFEL_quest_rs/Exploration/Ex08/../../work/data/ex08/corpus.txt
  input_format: 
  model_prefix: spm_bpe_chatbot
  model_type: BPE
  vocab_size: 4000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 999999
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: [UNK]
  bos_piece: [BOS]
  eos_pie


========= 🚀 학습을 개시합니다 =========
Epoch 01/10 | Avg Masked Loss: 5.6942
Epoch 02/10 | Avg Masked Loss: 4.6937
Epoch 03/10 | Avg Masked Loss: 4.0672
Epoch 04/10 | Avg Masked Loss: 3.5577
Epoch 05/10 | Avg Masked Loss: 3.1237
Epoch 06/10 | Avg Masked Loss: 2.7274
Epoch 07/10 | Avg Masked Loss: 2.3738
Epoch 08/10 | Avg Masked Loss: 2.0454
Epoch 09/10 | Avg Masked Loss: 1.7639
Epoch 10/10 | Avg Masked Loss: 1.5017

========= 🤖 챗봇 서비스 추론 테스트 =========
질문(Q): 오늘 너무 힘들다
답변(A): 힘내지 않아도 돼요 .
----------------------------------------
질문(Q): 10년 연애 끝 부질없네
답변(A): 더 좋은 사람 만날 수 있을 거예요 .
----------------------------------------
질문(Q): 인사해줘
답변(A): 많이 알면 도움이 되네요 .
----------------------------------------
질문(Q): 맛있는 거 추천해줘
답변(A): 그럴 수 있어요 .
----------------------------------------


In [ ]:
transformer_bot2 = Transformer(
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
    units=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT
)
train_transformer_with_train_earlystop(transformer_bot2, dataloader, epochs=20, lr=0.0005)




========= 🚀 Train 수렴 기준 Early Stopping 학습 시작 =========
Epoch 01/20 | Avg Masked Train Loss: 5.6856
 Train loss 감소함 (inf --> 5.685574). 가중치 백업 중...
Epoch 02/20 | Avg Masked Train Loss: 4.6803
 Train loss 감소함 (5.685574 --> 4.680286). 가중치 백업 중...
Epoch 03/20 | Avg Masked Train Loss: 4.0584
 Train loss 감소함 (4.680286 --> 4.058381). 가중치 백업 중...
Epoch 04/20 | Avg Masked Train Loss: 3.5543
 Train loss 감소함 (4.058381 --> 3.554261). 가중치 백업 중...
Epoch 05/20 | Avg Masked Train Loss: 3.1242
 Train loss 감소함 (3.554261 --> 3.124158). 가중치 백업 중...
Epoch 06/20 | Avg Masked Train Loss: 2.7299
 Train loss 감소함 (3.124158 --> 2.729935). 가중치 백업 중...
Epoch 07/20 | Avg Masked Train Loss: 2.3767
 Train loss 감소함 (2.729935 --> 2.376738). 가중치 백업 중...
Epoch 08/20 | Avg Masked Train Loss: 2.0512
 Train loss 감소함 (2.376738 --> 2.051196). 가중치 백업 중...
Epoch 09/20 | Avg Masked Train Loss: 1.7713
 Train loss 감소함 (2.051196 --> 1.771325). 가중치 백업 중...
Epoch 10/20 | Avg Masked Train Loss: 1.5098
 Train loss 감소함 (1.771325 --> 1.

In [47]:
# --- [Step 5: 예측 함수를 통한 모델 평가 및 테스트] ---
print("\n========= 🤖 챗봇 서비스 추론 테스트 2 =========")
for question in test_questions:
    response = evaluate_predict(question, transformer_bot2, tokenizer, max_len=MAX_LEN)
    print(f"질문(Q): {question}")
    print(f"답변(A): {response}")
    print("-" * 40)


========= 🤖 챗봇 서비스 추론 테스트 2 =========
질문(Q): 오늘 너무 힘들다
답변(A): 고생 많았어요 .
----------------------------------------
질문(Q): 10년 연애 끝 부질없네
답변(A): 더 좋은 사람 만나실 거예요 .
----------------------------------------
질문(Q): 인사해줘
답변(A): 나쁜 사람이니까요 .
----------------------------------------
질문(Q): 맛있는 거 추천해줘
답변(A): 솜씨가 좋으시네요 .
----------------------------------------


결론

형태소 분석기 없이 SentencePiece와 순수 파이토치 트랜스포머 알고리즘만으로 감정 맥락을 이해하는 답변 생성 능력을 확인했으나, 

완벽한 문장력 구사를 위해 추가 학습 및 추론 디코딩 전략 수정이 요구된다


In [45]:
import torch.optim.lr_scheduler as lr_scheduler

# --- [기능 1: 검토 완료된 학습률 람다 함수] ---
def get_lr_lambda(d_model, warmup_steps=4000):
    d_model = float(d_model)
    def lr_lambda(step):
        # step은 0부터 시작하므로 +1로 보정
        step = step + 1
        return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
    return lr_lambda


# --- [기능 2: 패딩 마스크 반영 정확도 함수] ---
def accuracy_function(y_pred, y_true, pad_id=0):
    """
    y_pred: (batch_size, seq_len, vocab_size)
    y_true: (batch_size, seq_len)
    """
    preds = y_pred.argmax(dim=-1)  # (batch_size, seq_len)
    mask = (y_true != pad_id)
    correct = (preds == y_true) & mask
    
    # 분모가 0이 되는 것을 방지
    total_valid_tokens = mask.float().sum()
    if total_valid_tokens == 0:
        return torch.tensor(0.0, device=y_pred.device)
        
    acc = correct.float().sum() / total_valid_tokens
    return acc


# --- [기능 3: 스케줄러 및 정확도가 통합된 학습 함수] ---
def train_transformer_v2(model, dataloader, epochs=20, d_model=256, warmup_steps=4000, patience=5):
    model.to(device)
    
    # 1. Optimizer 및 검토된 스케줄러 정의 (lr=1.0 베이스)
    optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_lambda(d_model, warmup_steps=warmup_steps))
    
    early_stopping = EarlyStopping(patience=patience, verbose=True)
    
    print("\n========= 🚀 스케줄러 & 정확도 탑재 학습 시작 =========")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_acc = 0
        
        for batch_idx, (inputs, dec_inputs, targets) in enumerate(dataloader):
            inputs, dec_inputs, targets = inputs.to(device), dec_inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs, dec_inputs)
            
            loss = loss_function(targets, outputs)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            # 파라미터 업데이트 및 배치 단위 스케줄러 스텝 업
            optimizer.step()
            scheduler.step()  # 💡 여기에 위치해야 매 step마다 lr이 갱신됩니다.
            
            # 메트릭 누적
            acc = accuracy_function(outputs, targets, pad_id=0)
            total_loss += loss.item()
            total_acc += acc.item()
            
        avg_train_loss = total_loss / len(dataloader)
        avg_train_acc = total_acc / len(dataloader)
        
        # 현재 적용 중인 실제 학습률 확인용
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_train_loss:.4f} | Acc: {avg_train_acc:.4f} | LR: {current_lr:.6f}")
        
        # Early Stopping 체크 (Train Loss 수렴 기준)
        early_stopping(avg_train_loss, model)
        if early_stopping.early_stop:
            print(f"🛑 조기 종료: Train Loss가 {patience} 에포크 동안 개선되지 않아 최적 수렴으로 종료합니다.")
            break
            
    if early_stopping.best_model_state is not None:
        model.load_state_dict(early_stopping.best_model_state)
        print("🎯 역대 최저 Loss 시점의 가중치 복원 완료.")

In [46]:
transformer_bot3 = Transformer(
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
    units=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT
)
train_transformer_v2(
        model=transformer_bot3,
        dataloader=dataloader,
        epochs=30,
        d_model=D_MODEL,
        warmup_steps=2000, # 데이터 크기에 맞춰 적절히 축소 조정 가능
        patience=4
)


========= 🚀 스케줄러 & 정확도 탑재 학습 시작 =========
Epoch 01/30 | Loss: 7.3181 | Acc: 0.1836 | LR: 0.000130
 Train loss 감소함 (inf --> 7.318113). 가중치 백업 중...
Epoch 02/30 | Loss: 5.5852 | Acc: 0.2587 | LR: 0.000259
 Train loss 감소함 (7.318113 --> 5.585156). 가중치 백업 중...
Epoch 03/30 | Loss: 5.0785 | Acc: 0.2824 | LR: 0.000389
 Train loss 감소함 (5.585156 --> 5.078535). 가중치 백업 중...
Epoch 04/30 | Loss: 4.5488 | Acc: 0.3134 | LR: 0.000518
 Train loss 감소함 (5.078535 --> 4.548761). 가중치 백업 중...
Epoch 05/30 | Loss: 4.0051 | Acc: 0.3583 | LR: 0.000647
 Train loss 감소함 (4.548761 --> 4.005145). 가중치 백업 중...
Epoch 06/30 | Loss: 3.4877 | Acc: 0.4035 | LR: 0.000776
 Train loss 감소함 (4.005145 --> 3.487682). 가중치 백업 중...
Epoch 07/30 | Loss: 3.0189 | Acc: 0.4515 | LR: 0.000906
 Train loss 감소함 (3.487682 --> 3.018854). 가중치 백업 중...
Epoch 08/30 | Loss: 2.5808 | Acc: 0.5058 | LR: 0.001035
 Train loss 감소함 (3.018854 --> 2.580810). 가중치 백업 중...
Epoch 09/30 | Loss: 2.2048 | Acc: 0.5595 | LR: 0.001164
 Train loss 감소함 (2.580810 --> 2.20

In [48]:
# --- [Step 5: 예측 함수를 통한 모델 평가 및 테스트] ---
print("\n========= 🤖 챗봇 서비스 추론 테스트 3 =========")
for question in test_questions:
    response = evaluate_predict(question, transformer_bot3, tokenizer, max_len=MAX_LEN)
    print(f"질문(Q): {question}")
    print(f"답변(A): {response}")
    print("-" * 40)


========= 🤖 챗봇 서비스 추론 테스트 3 =========
질문(Q): 오늘 너무 힘들다
답변(A): 고생 많았어요 .
----------------------------------------
질문(Q): 10년 연애 끝 부질없네
답변(A): 더 좋은 사람 만나실 거예요 .
----------------------------------------
질문(Q): 인사해줘
답변(A): 잘 먹고 잘 하는 건 어떨까요 .
----------------------------------------
질문(Q): 맛있는 거 추천해줘
답변(A): 먹고 알 먹고 없아가 있으니까요 .
----------------------------------------
